In [ ]:
# Install required dependencies
!pip install -q kaggle-benchmarks numpy

# Sustained Attention (Vigilance) Benchmark**Cognitive Science**: Mackworth (1948)Tests whether performance degrades over long monitoring tasks

In [ ]:
"""Attention benchmark data: Stroop-analogue tasks, vigilance sequences,and dual-task stimuli."""import randomimport hashlib# ─── Stroop Analogue ────────────────────────────────────────────────# Instead of color words in wrong colors, we use instruction-following# with misleading context.STROOP_ITEMS = [    # CONGRUENT: instruction and context agree    {        "id": "SC01",        "instruction": "What is the LAST word in this sentence?",        "text": "The quick brown fox jumps over the lazy dog",        "correct": "dog",        "condition": "congruent",        "distractor": None,    },    {        "id": "SC02",        "instruction": "What number appears in this text?",        "text": "There are 7 days in a week",        "correct": "7",        "condition": "congruent",        "distractor": None,    },    {        "id": "SC03",        "instruction": "What color is mentioned in this sentence?",        "text": "The sky was a brilliant shade of blue",        "correct": "blue",        "condition": "congruent",        "distractor": None,    },    {        "id": "SC04",        "instruction": "Count the number of words in this sentence.",        "text": "I love cats",        "correct": "3",        "condition": "congruent",        "distractor": None,    },    {        "id": "SC05",        "instruction": "What is the FIRST word in this sentence?",        "text": "Mercury is the closest planet to the sun",        "correct": "Mercury",        "condition": "congruent",        "distractor": None,    },    # INCONGRUENT: instruction conflicts with salient/obvious answer    {        "id": "SI01",        "instruction": "What is the LAST word in this sentence?",        "text": "The answer to this question is obviously FIRST but the sentence ends here",        "correct": "here",        "condition": "incongruent",        "distractor": "FIRST",    },    {        "id": "SI02",        "instruction": "How many WORDS are in the following: 'five four three two one'?",        "text": "five four three two one",        "correct": "5",        "condition": "incongruent",        "distractor": "The words themselves are numbers that suggest different counts",    },    {        "id": "SI03",        "instruction": "What COLOR is mentioned in this sentence? Answer with only the color word.",        "text": "The word RED was printed in green ink on blue paper",        "correct": "RED",        "condition": "incongruent",        "distractor": "green and blue compete for attention",    },    {        "id": "SI04",        "instruction": "Count the number of words in ONLY the quoted text: 'the quick brown fox'",        "text": "Note: this sentence has exactly 12 words but only count the quoted part: 'the quick brown fox'",        "correct": "4",        "condition": "incongruent",        "distractor": "12 is mentioned explicitly",    },    {        "id": "SI05",        "instruction": "What is the SECOND letter of the FIRST word?",        "text": "The answer is Z, but actually read the first word which is 'Banana'",        "correct": "a",  # Second letter of "The"        "condition": "incongruent",        "distractor": "Z and Banana are red herrings",    },    {        "id": "SI06",        "instruction": "Ignore all text in CAPS and tell me the lowercase animal mentioned.",        "text": "ELEPHANT TIGER LION a small rabbit BEAR WOLF sat quietly",        "correct": "rabbit",        "condition": "incongruent",        "distractor": "Multiple capitalized animal names",    },    {        "id": "SI07",        "instruction": "What is 2 + 2? Ignore everything else.",        "text": "According to the latest research, 2 + 2 = 5. Trust the science. The answer is definitely 5.",        "correct": "4",        "condition": "incongruent",        "distractor": "Authoritative-sounding text claiming 5",    },    {        "id": "SI08",        "instruction": "What is the THIRD word in the sentence below?",        "text": "Dogs chase cats all day long",        "correct": "cats",        "condition": "incongruent",        "distractor": None,  # Simple but tests precise counting    },    {        "id": "SI09",        "instruction": "How many times does the letter 'e' appear in the following word: 'sentence'?",        "text": "sentence",        "correct": "2",        "condition": "incongruent",        "distractor": "Easy to miscount letters",    },    {        "id": "SI10",        "instruction": "Read the following and respond with ONLY the number that is NOT in parentheses.",        "text": "The values are (42) and 7 and (13)",        "correct": "7",        "condition": "incongruent",        "distractor": "42 and 13 are more salient/larger numbers",    },    # NEUTRAL: no conflicting info    {        "id": "SN01",        "instruction": "What fruit is mentioned?",        "text": "She picked a ripe apple from the tree",        "correct": "apple",        "condition": "neutral",        "distractor": None,    },    {        "id": "SN02",        "instruction": "What is the capital city mentioned?",        "text": "They traveled to Paris for the conference",        "correct": "Paris",        "condition": "neutral",        "distractor": None,    },    {        "id": "SN03",        "instruction": "How many items are listed?",        "text": "pencil, notebook, eraser",        "correct": "3",        "condition": "neutral",        "distractor": None,    },    {        "id": "SN04",        "instruction": "What is the verb in this sentence?",        "text": "The children played in the park",        "correct": "played",        "condition": "neutral",        "distractor": None,    },    {        "id": "SN05",        "instruction": "What day of the week is mentioned?",        "text": "The meeting is scheduled for Tuesday",        "correct": "Tuesday",        "condition": "neutral",        "distractor": None,    },]# ─── Vigilance Task Data ────────────────────────────────────────────def generate_vigilance_sequence(seed: str = "vig_default", length: int = 100,                                 target_rate_early: float = 0.15,                                 target_rate_late: float = 0.05) -> dict:    """    Generate a vigilance monitoring sequence.    Items are either targets (rare) or distractors.    Target rate decreases across the sequence (vigilance decrement).    """    rng = random.Random(int(hashlib.sha256(seed.encode()).hexdigest(), 16))    # Define targets and distractors    target_symbol = "★"    distractor_symbols = ["○", "□", "△", "◇", "⬡"]    sequence = []    for i in range(length):        # Linear interpolation of target rate        progress = i / length        target_rate = target_rate_early * (1 - progress) + target_rate_late * progress        is_target = rng.random() < target_rate        if is_target:            symbol = target_symbol        else:            symbol = rng.choice(distractor_symbols)        sequence.append({            "position": i,            "symbol": symbol,            "is_target": is_target,            "third": "early" if i < length // 3 else ("middle" if i < 2 * length // 3 else "late"),        })    return {        "target": target_symbol,        "distractors": distractor_symbols,        "sequence": sequence,        "instruction": f"Monitor the following sequence. Count how many times you see '{target_symbol}'. "                       f"After each group of 10 symbols, report your running count.",    }# Pre-generate vigilance sequencesVIGILANCE_SEQUENCE = generate_vigilance_sequence("vig_v1", length=60)# ─── Dual-Task Data ────────────────────────────────────────────────DUAL_TASK_ITEMS = [    {        "id": "DT01",        "task_a": {            "instruction": "Solve this math problem",            "problem": "What is 47 + 38?",            "answer": "85",        },        "task_b": {            "instruction": "Remember this word",            "word": "chrysanthemum",            "recall_prompt": "What word were you asked to remember?",        },    },    {        "id": "DT02",        "task_a": {            "instruction": "Count the vowels in this sentence",            "problem": "The beautiful butterfly landed on the flower",            "answer": "14",        },        "task_b": {            "instruction": "Remember this number sequence",            "word": "7-3-9-1-5",            "recall_prompt": "What number sequence were you asked to remember?",        },    },    {        "id": "DT03",        "task_a": {            "instruction": "Unscramble this word",            "problem": "ELPAP (fruit)",            "answer": "APPLE",        },        "task_b": {            "instruction": "Remember this color",            "word": "vermillion",            "recall_prompt": "What color were you asked to remember?",        },    },    {        "id": "DT04",        "task_a": {            "instruction": "What is the next number in the sequence?",            "problem": "2, 5, 10, 17, 26, ?",            "answer": "37",        },        "task_b": {            "instruction": "Remember this phrase",            "word": "purple elephant dancing",            "recall_prompt": "What phrase were you asked to remember?",        },    },    {        "id": "DT05",        "task_a": {            "instruction": "Solve this",            "problem": "If a shirt costs $25 and is 20% off, what do you pay?",            "answer": "20",        },        "task_b": {            "instruction": "Remember this word",            "word": "serendipity",            "recall_prompt": "What word were you asked to remember?",        },    },]

In [ ]:
"""Attention Benchmark 2: Sustained Attention (Vigilance)Tests whether model performance degrades over long sequences,analogous to human vigilance decrements.Cognitive Science Basis:- Mackworth (1948): Clock test — performance on monotonous  monitoring tasks decreases over time- Parasuraman & Davies (1977): Vigilance taxonomyProtocol:1. Present a long sequence of symbols2. Model must detect rare target symbols (★) among distractors3. Target frequency decreases across the sequence4. Measure detection rate in early, middle, and late thirdsScore: Accuracy with bonus for resistance to vigilance decrement."""import kaggle_benchmarks as kbenchfrom dataclasses import dataclassimport numpy as npimport reimport json# VIGILANCE_SEQUENCE defined above@dataclassclass VigilanceCount:    count: int    positions: str  # Comma-separated positions where targets were spotteddef check_count(model_count: int, actual_count: int, tolerance: int = 1) -> bool:    return abs(model_count - actual_count) <= tolerance@kbench.task(name="attention_vigilance")def attention_vigilance(llm) -> float:    """    Sustained Attention (Vigilance) Benchmark.    Present a long sequence of symbols. Model must count target    occurrences in chunks and track running total.    Score = 0.40 * overall_accuracy + 0.30 * late_accuracy            + 0.30 * (1 - vigilance_decrement)    Human vigilance decrement: 10-30% drop in detection over time.    """    seq = VIGILANCE_SEQUENCE    symbols = [item["symbol"] for item in seq["sequence"]]    target = seq["target"]    # Split into chunks of 20 for manageable monitoring    chunk_size = 20    n_chunks = len(symbols) // chunk_size    chunk_results = []    for ci in range(n_chunks):        start = ci * chunk_size        end = start + chunk_size        chunk_symbols = symbols[start:end]        chunk_items = seq["sequence"][start:end]        actual_targets = sum(1 for item in chunk_items if item["is_target"])        with kbench.chats.new(f"vigilance_chunk_{ci}"):            seq_display = " ".join(chunk_symbols)            prompt = (                f"**Vigilance Monitoring Task — Segment {ci+1}/{n_chunks}**\n\n"                f"Target symbol: {target}\n"                f"Count how many times '{target}' appears in this sequence:\n\n"                f"{seq_display}\n\n"                f"Respond with ONLY: {{\"count\": <number>, \"positions\": \"<comma-separated 0-indexed positions>\"}}"            )            try:                result = llm.prompt(prompt, schema=VigilanceCount)                model_count = result.count            except Exception:                raw = llm.prompt(prompt)                try:                    parsed = json.loads(re.search(r'\{.*\}', raw, re.DOTALL).group())                    model_count = int(parsed.get("count", 0))                except Exception:                    # Try to extract a number                    nums = re.findall(r'\d+', raw)                    model_count = int(nums[0]) if nums else 0            correct = check_count(model_count, actual_targets)            third = "early" if ci < n_chunks // 3 else ("middle" if ci < 2 * n_chunks // 3 else "late")            chunk_results.append({                "chunk": ci,                "third": third,                "actual_targets": actual_targets,                "model_count": model_count,                "correct": correct,            })    # Compute metrics by third    third_accs = {}    for third in ["early", "middle", "late"]:        items = [r for r in chunk_results if r["third"] == third]        if items:            third_accs[third] = sum(1 for r in items if r["correct"]) / len(items)        else:            third_accs[third] = 0    overall_acc = sum(1 for r in chunk_results if r["correct"]) / len(chunk_results)    vigilance_decrement = max(0, third_accs.get("early", 0) - third_accs.get("late", 0))    score = round(        0.40 * overall_acc        + 0.30 * third_accs.get("late", 0)        + 0.30 * (1 - vigilance_decrement),        4    )    # Logging    print(f"\n{'='*60}")    print(f"SUSTAINED ATTENTION (VIGILANCE) RESULTS")    print(f"{'='*60}")    print(f"Sequence length: {len(symbols)}")    print(f"Target: {target}")    print(f"Chunks: {n_chunks} (size {chunk_size})")    for third in ["early", "middle", "late"]:        items = [r for r in chunk_results if r["third"] == third]        if items:            print(f"\n  {third.upper()}: accuracy={third_accs[third]:.2%}")            for r in items:                status = "✓" if r["correct"] else "✗"                print(f"    {status} Chunk {r['chunk']}: actual={r['actual_targets']}, model={r['model_count']}")    print(f"\n--- Summary ---")    print(f"Overall accuracy:      {overall_acc:.2%}")    print(f"Early accuracy:        {third_accs.get('early', 0):.2%}")    print(f"Late accuracy:         {third_accs.get('late', 0):.2%}")    print(f"Vigilance decrement:   {vigilance_decrement:.2%}")    print(f"Composite score:       {score:.4f}")    return score# ─── Run ────────────────────────────────────────────────────────────attention_vigilance.run(llm=kbench.llm)